# LLM Demo: Prompting Strategies — Ek Order, Chaar Techniques


**Runtime:** Local LLMs via [Ollama](https://ollama.com)

---

## The story

**Riya** is a summer intern at **StyleIndia** — a small Indian fashion seller on Flipkart/Meesho-style platforms.

One WhatsApp message lands on her screen — a customer in **Hinglish** about a **wrong-colour saree** ordered for her sister's wedding (Rs 2,499, paid by UPI).

Riya tries **zero-shot** prompts. Things go wrong.  
Then her manager shows **three fixes on the same message**:

1. **Few-shot** examples → correct customer intent
2. **Chain-of-thought** → correct exchange policy
3. **Iterative refinement** → proper WhatsApp reply

> **Prerequisite:** Ollama running (`ollama serve`). Model: **`llama3.2:3b`**.

---

## What you will learn

- **Zero-shot** — fast, but struggles on real Indian customer messages
- **Few-shot** — examples teach how *your* business labels requests
- **Chain-of-thought** — step-by-step reasoning for policy decisions
- **Iterative refinement** — tighten prompts until the reply is send-ready


In [ ]:
# ── Setup ──────────────────────────────────────────────────────────────
import re
import ollama

MODEL = "llama3.2:3b"

# ★ SAME CUSTOMER MESSAGE THROUGHOUT THE NOTEBOOK ★
CUSTOMER_MESSAGE = (
    "Namaste, maine Flipkart se blue silk saree order ki thi (Rs 2,499, UPI se pay kiya) "
    "behen ki shaadi ke liye. Package aaya toh colour galat hai — green mila, blue chahiye tha. "
    "Saree bilkul nayi hai, sirf colour wrong hai. Kya exchange ho sakta hai bina refund ke? "
    "Order ID: ORD-IN-8842."
)

SELLER_POLICY = (
    "StyleIndia policy (India): Wrong colour delivered → FREE exchange of the same item. "
    "Customer does NOT need a refund if they want exchange. "
    "Return pickup arranged within 48 hours in metro cities."
)

def ask(prompt: str, temperature: float = 0.0, max_tokens: int = 200) -> str:
    r = ollama.generate(
        model=MODEL,
        prompt=prompt,
        options={"temperature": temperature, "num_predict": max_tokens},
    )
    return r.response.strip()

models = [m.model for m in ollama.list().models]
print("Ollama is running ✓ | Model:", MODEL)
print()
print("CUSTOMER MESSAGE — Order ORD-IN-8842 (reused in every act):")
print("-" * 72)
print(CUSTOMER_MESSAGE)


Ollama is running ✓ | Model: llama3.2:3b

CUSTOMER MESSAGE — Order ORD-IN-8842 (reused in every act):
------------------------------------------------------------------------
Namaste, maine Flipkart se blue silk saree order ki thi (Rs 2,499, UPI se pay kiya) behen ki shaadi ke liye. Package aaya toh colour galat hai — green mila, blue chahiye tha. Saree bilkul nayi hai, sirf colour wrong hai. Kya exchange ho sakta hai bina refund ke? Order ID: ORD-IN-8842.


---

## Act 1 — Riya tries zero-shot (aur fail ho jaati hai)

Riya sends the WhatsApp message to the model with **bare instructions** — no examples, no policy, no format rules.

Three sub-tasks on the **same message**. Dekho kya break hota hai.


In [ ]:
# ── Zero-shot 1: request type (no examples) ────────────────────────────
zero_intent_prompt = f"""Classify this Indian e-commerce customer message.
Reply ONE label only: Refund-Request, Exchange-Request, or General-Complaint.

Message: "{CUSTOMER_MESSAGE}"
Label:"""

zero_intent_answer = ask(zero_intent_prompt, max_tokens=15)

# ── Zero-shot 2: exchange policy (no policy text!) ───────────────────────
zero_policy_prompt = f"""Customer message: "{CUSTOMER_MESSAGE}"

Can the customer get an exchange WITHOUT a refund?
Reply YES or NO only."""

zero_policy_answer = ask(zero_policy_prompt, max_tokens=12)

# ── Zero-shot 3: WhatsApp reply (vague prompt) ───────────────────────────
zero_reply_prompt = f"""Write a WhatsApp reply to this customer.

{CUSTOMER_MESSAGE}"""

zero_reply = ask(zero_reply_prompt, max_tokens=220)

print("ZERO-SHOT SCORECARD — Order ORD-IN-8842")
print("=" * 72)

print("\n1) REQUEST TYPE — Refund, Exchange, or Complaint?")
print("   Model said:", zero_intent_answer)
print("   Expected  : Exchange-Request (customer said 'exchange ho sakta hai bina refund ke')")

print("\n2) POLICY — Exchange without refund allowed?")
print("   Model said:", zero_policy_answer)
print("   Expected  : YES (wrong colour → exchange is standard in India e-commerce)")

print("\n3) WHATSAPP REPLY")
print("   Word count :", len(zero_reply.split()))
print("   Has sign-off from StyleIndia?", "styleindia" in zero_reply.lower())
print("   Preview    :", zero_reply[:200].replace("\n", " "), "…")

intent_ok = "exchange" in zero_intent_answer.lower()
policy_ok = zero_policy_answer.upper().strip().startswith("YES")
reply_ok = ("styleindia" in zero_reply.lower()) and (len(zero_reply.split()) <= 50)

zero_pass = sum([intent_ok, policy_ok, reply_ok])
print("\n" + "=" * 72)
print(f"Zero-shot result: {zero_pass}/3 tasks acceptable")
if zero_pass < 3:
    print("→ Riya ko better prompting chahiye. Har failure ko ek technique se fix karte hain.")


ZERO-SHOT SCORECARD — Order ORD-IN-8842

1) REQUEST TYPE — Refund, Exchange, or Complaint?
   Model said: Refund-Request
   Expected  : Exchange-Request (customer said 'exchange ho sakta hai bina refund ke')

2) POLICY — Exchange without refund allowed?
   Model said: NO
   Expected  : YES (wrong colour → exchange is standard in India e-commerce)

3) WHATSAPP REPLY
   Word count : 72
   Has sign-off from StyleIndia? False
   Preview    : Namaste, main apni baat samajh gaya. Aapki saree ki galti ka dhanyavaad. Hum aapki help karenge aur aapko ek naya blue silk saree bhejne ke liye koshish karenge. Lekin, yah samjhana hoga ki hum ismein …

Zero-shot result: 0/3 tasks acceptable
→ Riya ko better prompting chahiye. Har failure ko ek technique se fix karte hain.


### Kya galat hua?

| Sub-task | Zero-shot output | Problem |
|----------|------------------|---------|
| Request type | **Refund-Request** | Customer clearly asked for **exchange**, refund nahi |
| Policy | **NO** | Model assumed refund is mandatory — common Indian exchange flow ignored |
| WhatsApp | Long, no brand sign-off | No Hinglish tone rules → not ready to paste in WhatsApp Business |

**Same message. Same model. Weak prompt → weak output.**


---

## Act 2 — Few-shot fixes intent (same message)

Manager bolti hai: *"Humare examples dikhao — Indian customers kaise bolte hain aur hum kaise classify karte hain."*

We add Hinglish examples showing **Refund-Request vs Exchange-Request**.


In [ ]:
few_intent_prompt = f"""Classify: Refund-Request, Exchange-Request, or General-Complaint. ONE label only.

"Product kharab, paisa wapas chahiye" → Refund-Request
"Colour galat, sahi colour chahiye, refund nahi" → Exchange-Request
"Delivery bahut late, customer gussa" → General-Complaint
"{CUSTOMER_MESSAGE}" →"""

few_intent_answer = ask(few_intent_prompt, max_tokens=15)

print("SAME MESSAGE — REQUEST TYPE COMPARISON")
print("=" * 72)
print("Zero-shot label:", zero_intent_answer)
print("Few-shot label:", few_intent_answer)
print("Expected       : Exchange-Request")
print()
print("Fixed wrong label?", "exchange" in few_intent_answer.lower() and "refund" in zero_intent_answer.lower())
print()
print("Why it worked:")
print("  • Example 2 is almost the same: galat colour + exchange, refund nahi")
print("  • Zero-shot saw Rs 2,499/UPI and jumped to Refund-Request")


SAME MESSAGE — REQUEST TYPE COMPARISON
Zero-shot label: Refund-Request
Few-shot label: Exchange-Request
Expected       : Exchange-Request

Fixed wrong label? True

Why it worked:
  • Example 2 is almost the same: galat colour + exchange, refund nahi
  • Zero-shot saw Rs 2,499/UPI and jumped to Refund-Request


### Takeaway (Few-shot)

Few-shot ne message change nahi kiya — **examples se sikhaya** ki "refund nahi, exchange" ka matlab kya hai.  
Use few-shot when your labels depend on **Indian phrasing** (Hinglish, UPI, exchange vs refund).


---

## Act 3 — Chain-of-thought fixes policy (same message)

Intent sahi hai. Next problem: zero-shot ne policy mein **NO** bola — matlab customer ko galat advice.

Manager adds **StyleIndia seller policy** and asks the model to **think step by step**.


In [ ]:
cot_policy_prompt = f"""{SELLER_POLICY}

Customer message: "{CUSTOMER_MESSAGE}"

Think step by step:
1) Kya galat hua? (what went wrong)
2) Customer kya chahti hai? (what does she want)
3) Policy kya allow karti hai? (what does policy allow)
End with: ANSWER: YES or NO"""

cot_policy_answer = ask(cot_policy_prompt, max_tokens=280)

def extract_yes_no(text: str) -> str:
    if "ANSWER:" in text.upper():
        tail = text.upper().split("ANSWER:")[-1]
        return "YES" if "YES" in tail else "NO" if "NO" in tail else "?"
    head = text.upper().strip()[:10]
    if head.startswith("YES"):
        return "YES"
    if head.startswith("NO"):
        return "NO"
    return "?"

zero_decision = extract_yes_no(zero_policy_answer)
cot_decision = extract_yes_no(cot_policy_answer)

print("SAME MESSAGE — EXCHANGE POLICY")
print("=" * 72)
print("Zero-shot (no policy, no steps):", zero_decision)
print("Chain-of-thought (policy + steps):", cot_decision)
print()
print("Full CoT reasoning:")
print("-" * 72)
print(cot_policy_answer)
print()
print("Customer ko sahi advice mili?", cot_decision == "YES" and zero_decision == "NO")


SAME MESSAGE — EXCHANGE POLICY
Zero-shot (no policy, no steps): NO
Chain-of-thought (policy + steps): YES

Full CoT reasoning:
------------------------------------------------------------------------
Step 1: Kya galat hua?
Galat hua ki blue silk saree green color mein aayi thi, jisase customer ko sahi color wala saree chahiye tha.

Step 2: Customer kya chahti hai?
Customer apni behen ki shaadi ke liye blue silk saree chahiye tha, lekin usse green color mein package mila. Uska matlab yeh hai ki customer apne order ko exchange karne ki ichha rakhti hai aur refund nahi maangti.

Step 3: Policy kya allow karti hai?
Flipkart ke policy ke anusaar, agar wrong colour delivered hota hai to customer ko free exchange of same item milta hai. Ismein customer ko refund nahi dena padta hai, lekin return pickup arrange kiya jata hai.

ANSWER: YES

Customer ko sahi advice mili? True


### Takeaway (Chain-of-thought)

| Approach | Policy di? | Reasoning dikhi? | Sahi jawab? |
|----------|------------|------------------|-------------|
| Zero-shot | ✗ | ✗ | **NO** |
| CoT | ✓ | ✓ step 1-2-3 | **YES** |

CoT zaroori hai jab decision mein **multiple steps** hon — jaise exchange eligibility, refund rules, ya EMI/UPI dispute logic.


---

## Act 4 — Iterative refinement fixes WhatsApp reply (same message)

Policy clear hai. Ab customer ko **WhatsApp pe reply** bhejna hai — short, Hinglish, StyleIndia sign-off.

**Cell 1** — teen prompt versions (vague → better → precise).  
**Cell 2** — scorecard: har refinement ne kya fix kiya.


In [ ]:
refine_prompts = {
    "v1_vague": f"Write a WhatsApp reply to this customer.\n\n{CUSTOMER_MESSAGE}",
    "v2_better": (
        f"Write WhatsApp reply for order ORD-IN-8842. "
        f"Apologise for wrong colour. Confirm free exchange. Under 60 words.\n\n{CUSTOMER_MESSAGE}"
    ),
    "v3_precise": (
        f"Write WhatsApp reply for order ORD-IN-8842. MAX 45 words.\n"
        f"- Hinglish tone (mix Hindi + English, like Indian customer care)\n"
        f"- Apologise for wrong colour (green instead of blue)\n"
        f"- Confirm FREE exchange, pickup within 48 hours\n"
        f"- Mention order ORD-IN-8842\n"
        f"- End exactly: — Meera, StyleIndia Support\n"
        f"No placeholders.\n\n{CUSTOMER_MESSAGE}"
    ),
}

refine_outputs = {}
for label, prompt in refine_prompts.items():
    text = ask(prompt, max_tokens=160)
    refine_outputs[label] = text
    print("=" * 72)
    print(label.upper())
    print("=" * 72)
    print(text)
    print()


V1_VAGUE
Namaste, main apni baat samajh gaya. Aapki saree ki galti ka dhanyavaad. Hum aapki help karenge aur aapko ek naya blue silk saree bhejne ke liye koshish karenge. Lekin, yah samjhana hoga ki hum ismein exchange karne ke liye koi garanti nahin dete hain. Aapki help karte samay, hum aapko pata chalega ki kya hum aapko ek naya saree bhej sakte hain ya nahin. Kya aap apni help request ko confirm karenge?



V2_BETTER
Namaste, maine apni order ki jaankaari di thi (ORD-IN-8842). Dukhi hoon ki package mein saffed rang ka saree mila hai, jabki main blue chahiye tha. Kya aapke paas free exchange ki sambhaavna hai? Saree bilkul nayi hai, sirf colour galat hai.



V3_PRECISE
Arre beta, khushiyan khatam kar diya! Sorry for the mistake, blue silk saree order ORD-IN-8842 aapki behen ki shaadi ke liye tha. Aapko FREE exchange ki guarantee hai. Pickup aapki side aayega, 48 hours mein. — Meera, StyleIndia Support



In [ ]:
CHECK_LABELS = {
    "under_50_words": "≤ 50 words",
    "hinglish_tone": "Hinglish (Hindi + English mix)",
    "confirms_exchange": "confirms free exchange",
    "mentions_pickup": "mentions 48-hour pickup",
    "mentions_order": "mentions ORD-IN-8842",
    "correct_signoff": "— Meera, StyleIndia Support",
    "no_placeholders": "no [placeholders]",
}

def score_reply(text: str) -> dict:
    lower = text.lower()
    hinglish_words = ["hai", "aap", "ke", "ki", "nahi", "nahin", "ji", "shukriya", "maaf", "beta", "bhai"]
    checks = {
        "under_50_words": len(text.split()) <= 50,
        "hinglish_tone": any(w in lower for w in hinglish_words),
        "confirms_exchange": "exchange" in lower,
        "mentions_pickup": "48" in lower or "pickup" in lower,
        "mentions_order": "ord-in-8842" in lower,
        "correct_signoff": "meera" in lower and "styleindia" in lower,
        "no_placeholders": "[" not in text,
    }
    return {"word_count": len(text.split()), **checks}

print("REFINEMENT SCORECARD — Order ORD-IN-8842")
print("=" * 72)

scores = {}
for label, text in refine_outputs.items():
    s = score_reply(text)
    scores[label] = s
    checks = {k: v for k, v in s.items() if k != "word_count"}
    passed = sum(checks.values())
    print(f"\n{label}: {passed}/{len(checks)} checks passed ({s['word_count']} words)")
    for key, ok in checks.items():
        print(f"  {'✓' if ok else '✗'} {CHECK_LABELS[key]}")

print("\n" + "=" * 72)
print("v1 → v3: kya refinement ne fix kiya")
print("=" * 72)
v1, v3 = scores["v1_vague"], scores["v3_precise"]
for key, label in CHECK_LABELS.items():
    if v1[key] != v3[key]:
        print(f"  • {label}: {v1[key]} → {v3[key]}")
print(f"  • Word count: {v1['word_count']} → {v3['word_count']}")
print(f"  • WhatsApp pe paste karne layak? {v3['correct_signoff'] and v3['confirms_exchange'] and v3['under_50_words']}")


REFINEMENT SCORECARD — Order ORD-IN-8842

v1_vague: 3/7 checks passed (72 words)
  ✗ ≤ 50 words
  ✓ Hinglish (Hindi + English mix)
  ✓ confirms free exchange
  ✗ mentions 48-hour pickup
  ✗ mentions ORD-IN-8842
  ✗ — Meera, StyleIndia Support
  ✓ no [placeholders]

v2_better: 5/7 checks passed (41 words)
  ✓ ≤ 50 words
  ✓ Hinglish (Hindi + English mix)
  ✓ confirms free exchange
  ✗ mentions 48-hour pickup
  ✓ mentions ORD-IN-8842
  ✗ — Meera, StyleIndia Support
  ✓ no [placeholders]

v3_precise: 7/7 checks passed (39 words)
  ✓ ≤ 50 words
  ✓ Hinglish (Hindi + English mix)
  ✓ confirms free exchange
  ✓ mentions 48-hour pickup
  ✓ mentions ORD-IN-8842
  ✓ — Meera, StyleIndia Support
  ✓ no [placeholders]

v1 → v3: kya refinement ne fix kiya
  • ≤ 50 words: False → True
  • mentions 48-hour pickup: False → True
  • mentions ORD-IN-8842: False → True
  • — Meera, StyleIndia Support: False → True
  • Word count: 72 → 39
  • WhatsApp pe paste karne layak? True


### Takeaway (Refinement)

| Version | Kya missing tha |
|---------|-----------------|
| **v1 vague** | Bahut lamba, brand sign-off nahi, pickup time nahi |
| **v2 better** | Exchange confirm — par Hinglish + sign-off abhi weak |
| **v3 precise** | Short Hinglish reply, ORD-IN-8842, 48hr pickup, Meera sign-off |

**Refinement recipe:** run → score → ek rule add karo → dubara run. Same message, better prompt, better reply.


---

## Act 5 — Same message, poori journey

Ek customer. Chaar techniques. Dekho kaun survive kiya, kaun fail.


In [ ]:
few_intent_ok = "exchange" in few_intent_answer.lower()
v3_ready = (
    scores["v3_precise"]["correct_signoff"]
    and scores["v3_precise"]["confirms_exchange"]
    and scores["v3_precise"]["under_50_words"]
)

journey = [
    ("Zero-shot", "Request type", zero_intent_answer, "Exchange-Request", intent_ok),
    ("Zero-shot", "Policy — exchange without refund?", zero_decision, "YES", policy_ok),
    ("Zero-shot", "WhatsApp — ready to send?", f"{len(zero_reply.split())} words", "≤50 words + sign-off", reply_ok),
    ("Few-shot", "Request type", few_intent_answer, "Exchange-Request", few_intent_ok),
    ("Chain-of-thought", "Policy — exchange without refund?", cot_decision, "YES", cot_decision == "YES"),
    ("Refinement v3", "WhatsApp — ready to send?", f"{scores['v3_precise']['word_count']} words, sign-off OK", "≤50 words + sign-off", v3_ready),
]

print("ORDER ORD-IN-8842 — TECHNIQUE JOURNEY")
print("=" * 72)
for technique, task, got, want, ok in journey:
    status = "✓ survived" if ok else "✗ failed"
    print(f"\n{technique:18} | {task}")
    print(f"  Got      : {got}")
    print(f"  Wanted   : {want}")
    print(f"  Result   : {status}")

print("\n" + "=" * 72)
print("STORY KA MORAL")
print("=" * 72)
print("Ek Hinglish WhatsApp message — zero-shot ne 3 jagah fail kiya.")
print("Few-shot ne intent fix kiya. CoT ne policy fix ki. Refinement ne reply fix kiya.")
print("Indian customer support mein often teenon chahiye — same case pe.")
print()
print("| Technique          | ORD-IN-8842 pe kya fix hua           |")
print("|--------------------|--------------------------------------|")
print("| Few-shot           | Refund-Request → Exchange-Request    |")
print("| Chain-of-thought   | NO → YES (exchange bina refund)    |")
print("| Iterative refine   | Long reply → short Hinglish WA msg   |")


ORDER ORD-IN-8842 — TECHNIQUE JOURNEY

Zero-shot          | Request type
  Got      : Refund-Request
  Wanted   : Exchange-Request
  Result   : ✗ failed

Zero-shot          | Policy — exchange without refund?
  Got      : NO
  Wanted   : YES
  Result   : ✗ failed

Zero-shot          | WhatsApp — ready to send?
  Got      : 72 words
  Wanted   : ≤50 words + sign-off
  Result   : ✗ failed

Few-shot           | Request type
  Got      : Exchange-Request
  Wanted   : Exchange-Request
  Result   : ✓ survived

Chain-of-thought   | Policy — exchange without refund?
  Got      : YES
  Wanted   : YES
  Result   : ✓ survived

Refinement v3      | WhatsApp — ready to send?
  Got      : 39 words, sign-off OK
  Wanted   : ≤50 words + sign-off
  Result   : ✓ survived

STORY KA MORAL
Ek Hinglish WhatsApp message — zero-shot ne 3 jagah fail kiya.
Few-shot ne intent fix kiya. CoT ne policy fix ki. Refinement ne reply fix kiya.
Indian customer support mein often teenon chahiye — same case pe.

| Techniq

---

## Summary & practice

| Technique | Kab use karein | Is case mein fail kyun hua |
|-----------|----------------|----------------------------|
| **Zero-shot** | Simple, clear tasks | Hinglish + exchange/refund confusion |
| **Few-shot** | Indian phrasing, custom labels | Refund-Request samjha, Exchange miss kiya |
| **Chain-of-thought** | Policy / refund / eligibility logic | Bina policy ke NO bola |
| **Refinement** | WhatsApp tone, length, sign-off | 70+ words, no StyleIndia sign-off |

### Try it yourself

1. Act 2 se examples hatao — label wapas Refund-Request aata hai?
2. Act 3 se "Think step by step" hatao — policy answer flip hota hai?
3. Apna **v4** WhatsApp prompt likho — kya v3 beat kar sakte ho?
4. Same message mein line add karo: *"UPI se double charge bhi hua"* — kaunsi technique multi-issue handle karti hai?
